# Week 4 - Feature Engineering

**Goal:** Build a non-leaking feature for the dataset

In this workbook, you will use a gridded **sea surface temperature (SST)** dataset from oceanography. SST varies smoothly across space and seasonally through time, so nearby observations are not independent.

**Dataset:** NOAA Extended Reconstructed Sea Surface Temperature version 5, accessed through `xarray`'s tutorial datasets. NOAA describes ERSSTv5 as a global monthly SST analysis on a 2-degree grid derived from ICOADS, with records extending from 1854 to the present. See:  
https://www.psl.noaa.gov/data/gridded/data.noaa.ersst.v5.html  
https://docs.xarray.dev/en/stable/generated/xarray.tutorial.open_dataset.html

## 1. Setup

Run the cell below. If any package is missing, uncomment the install line and run it once.

This notebook intentionally gives you some complete code and some incomplete code. Sections marked **TODO** are places where you should write or modify code yourself.

In [ ]:
# Uncomment if needed:
# !pip install numpy pandas xarray netcdf4 scikit-learn matplotlib

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import KFold, GroupKFold, cross_validate
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
pd.set_option("display.max_columns", 50)

## 2. Load the sea surface temperature dataset

The data are stored as an `xarray.Dataset`, which is common for gridded environmental and oceanographic data. Instead of rows and columns only, the data have dimensions such as `time`, `lat`, and `lon`.

The SST variable is measured over a regular latitude-longitude grid through time.

In [ ]:
ds = xr.tutorial.load_dataset("ersstv5")
ds

## 3. Subset the data

The full global dataset is larger than we need for this activity. The cell below selects a broad Pacific Ocean region and thins the grid and time axis so the model runs quickly.

We are still keeping enough data to preserve spatial and temporal correlation.

In [ ]:
# Select a broad Pacific region and a recent multi-decade period.
# The latitude coordinate is ordered from north to south, so slice(60, -60) is intentional.
ds_sub = (
    ds
    .sel(time=slice("1992-01-01", "2020-12-31"), lat=slice(60, -60), lon=slice(120, 290))
    .isel(lat=slice(None, None, 2), lon=slice(None, None, 2), time=slice(None, None, 3))
)

sst = ds_sub["sst"]
sst

## 4. Explore the data visually

Before modeling, inspect the target variable. SST has strong geographic structure and seasonal/temporal structure.

The first plot shows the SST field for one time step. Nearby ocean cells usually have similar temperatures, which is spatial autocorrelation.

In [ ]:
one_time = sst.isel(time=0)

fig, ax = plt.subplots(figsize=(10, 5))
one_time.plot(ax=ax)
ax.set_title(f"Sea surface temperature on {pd.to_datetime(one_time.time.values).date()}")
plt.show()

### Explore change through time

The next plot averages SST across the selected region for each time step. Because we kept every third month, the pattern will still show seasonal and long-term variation.

In [ ]:
regional_mean = sst.mean(dim=["lat", "lon"], skipna=True)

fig, ax = plt.subplots(figsize=(10, 4))
regional_mean.plot(ax=ax)
ax.set_title("Regional mean SST over time")
ax.set_ylabel("SST")
plt.show()

## 5. Convert gridded data to a machine-learning table

Most `sklearn` estimators expect a table where each row is one observation. Here, one row will represent one grid cell at one time.

The target is `sst`. The predictors will be simple features based on location and time.

We create cyclic features for month and longitude because December is close to January, and longitude wraps around the globe.

In [ ]:
df = sst.to_dataframe().reset_index().dropna(subset=["sst"]).copy()

# Time features
df["year"] = df["time"].dt.year
df["month"] = df["time"].dt.month

#TODO Create Month sin and cos features
df["month_sin"] = #TODO
df["month_cos"] = #TODO

print(df.shape)
df.head()

## 6. Define the prediction problem

We will predict SST from simple location and time features.

This is not meant to be the best possible ocean forecasting model. The goal is to study how evaluation changes when data are spatially and temporally correlated.

In [ ]:
feature_cols = [
    "lat",
    "lon",
    "month_sin",
    "month_cos",
    "year",
]

target_col = "sst"

X = df[feature_cols]
y = df[target_col]

X.head()

## 7. Create a reusable model and evaluation function

We use a tree-based regression model from `sklearn`. It can capture nonlinear patterns such as warmer temperatures near the equator and seasonal variation.

The helper function below runs cross-validation and returns RMSE, MAE, and R² for each fold.

Interpretation of metrics:

- **RMSE:** Typical prediction error, with larger errors penalized more strongly.
- **MAE:** Typical absolute error.
- **R²:** Fraction of variance explained. Higher is better, but it can be misleading when validation folds are not independent.

In [ ]:
def make_model():
    return Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("regressor", HistGradientBoostingRegressor(
            max_iter=100,
            learning_rate=0.08,
            max_leaf_nodes=31,
            random_state=42,
        ))
    ])


def summarize_cv(model, X, y, cv, groups=None):
    """Run cross-validation and return fold-level metrics."""
    scores = cross_validate(
        model,
        X,
        y,
        cv=cv,
        groups=groups,
        scoring={
            "rmse": "neg_root_mean_squared_error",
            "mae": "neg_mean_absolute_error",
            "r2": "r2",
        },
        n_jobs=-1,
        return_train_score=False,
    )

    results = pd.DataFrame({
        "fold": np.arange(1, len(scores["test_rmse"]) + 1),
        "RMSE": -scores["test_rmse"],
        "MAE": -scores["test_mae"],
        "R2": scores["test_r2"],
    })
    return results


def summarize_mean(results, label):
    """Summarize fold-level CV results into one row."""
    return pd.DataFrame({
        "Evaluation": [label],
        "Mean RMSE": [results["RMSE"].mean()],
        "SD RMSE": [results["RMSE"].std()],
        "Mean MAE": [results["MAE"].mean()],
        "Mean R2": [results["R2"].mean()],
    })

## 8. Naive evaluation: random k-fold cross-validation

Run the cell below and record the results.

In [ ]:
random_cv = KFold(n_splits=5, shuffle=True, random_state=42)

random_results = summarize_cv(
    model=make_model(),
    X=X,
    y=y,
    cv=random_cv,
)

random_results

In [ ]:
random_summary = summarize_mean(random_results, "Random k-fold CV")
random_summary

## 9. Feature engineering

TODO: Add historic monthly average temperture for each location  
**Be careful** to not use future data when creating this column  
Use `shift()` and `expanding()` to get cumulative average sst. 

In [ ]:
df['Month_Average_sst'] = #TODO

In [ ]:
feature_cols = [
    "lat",
    "lon",
    "month_cos",
    "month_sin",
    "year",
    "Month_Average_sst"
]

X = df[feature_cols]
y = df[target_col]

X.head()

In [ ]:
random_cv = KFold(n_splits=5, shuffle=True, random_state=42)

random_results = summarize_cv(
    model=make_model(),
    X=X,
    y=y,
    cv=random_cv,
)

random_results

In [ ]:
random_summary = summarize_mean(random_results, "Random k-fold CV")
random_summary